# Chapter 1: The Single Decision Tree — A Complete Deep Dive

> **Chapter Goal:** Chapter 0 taught us how a single node chooses its split. This chapter zooms out to the full tree — how it grows, why it fails, the tools we use to control it, and critically, **how to explain a tree's prediction** to a business stakeholder. This knowledge is required to understand why Random Forest and GBM were invented.

---

## 1. Three Algorithms: ID3, C4.5, and CART

A "Decision Tree" is not one algorithm — it's a family of algorithms that all follow the same high-level idea but differ in the *details* of how they split, what data they handle, and how the tree is shaped.

| Algorithm | Year | Metric | Split Type | Data Types Supported |
| :--- | :---: | :--- | :--- | :--- |
| **ID3** | 1986 | Information Gain | Multi-way | Categorical only |
| **C4.5** | 1993 | Gain Ratio | Multi-way | Categorical & Continuous |
| **CART** | 1984 | Gini (Class) / MSE (Regression) | **Binary only** | Categorical & Continuous |

### **Why does CART only do binary splits?**
ID3 and C4.5 do multi-way splits. For example, a "Color" feature with values `{Red, Blue, Green}` creates **3 branches** — one for each value.

CART forces every split to be binary: *left or right*. Everything must be framed as `feature <= threshold` (left) or `feature > threshold` (right). Even for categorical features (`Color = Red` vs. `Color != Red`).

**Why is binary better?**
1. **More flexible:** A multi-way split uses a feature's entire vocabulary in one shot. CART can ask about the same feature multiple times at different levels, using it more efficiently.
2. **Simpler tree traversal:** A binary tree is simpler to implement, cache, and optimize. Every internal node has exactly 2 children — no special cases.
3. **Better for continuous features:** Continuous features like `Age` have infinite possible values. A multi-way split has no meaning for them. Binary threshold splits (`Age > 35`) are natural and directly supported.

**Scikit-Learn uses CART.** All hyperparameters you set in `DecisionTreeClassifier` and `DecisionTreeRegressor` in Scikit-Learn are CART hyperparameters.

---

## 2. Regression Trees: Predicting Numbers Instead of Classes

### **How the Output Changes**
In Classification, a leaf predicts the **Majority Class** among the samples that reach it. In Regression, it predicts the **Mean (Average)** of all target values of the samples that reach it.

| Property | Classification Tree | Regression Tree |
| :--- | :--- | :--- |
| **Target** | Discrete label | Continuous number |
| **Leaf Predicts** | Majority class | Mean of samples |
| **Split Criterion** | Gini / Entropy | MSE / MAE |
| **Boundary Shape** | Step-wise class regions | Step-wise numeric approximation |

### **The Splitting Criterion: MSE Reduction**
We can't use "purity" for numbers (what would 100% pure salary mean?). Instead we use **variance reduction**. A good split groups similar numbers together and separates dissimilar ones.

For a node with $N$ samples and target values $y_1, y_2, ..., y_N$:

$$MSE(Node) = \frac{1}{N} \sum_{i=1}^{N} (y_i - \bar{y})^2$$

Where $\bar{y} = \frac{1}{N}\sum y_i$ is the mean of all target values in that node.

**The split cost** — the weighted MSE of both children:
$$Cost(Split) = \frac{N_{left}}{N} \cdot MSE_{left} + \frac{N_{right}}{N} \cdot MSE_{right}$$

The algorithm picks the split that **minimizes this weighted cost**.

### **✍️ Worked Example: Finding the Best Regression Split**

**Dataset** — Salary (K$): `[10, 20, 30, 80, 90]`, sorted. One feature: Experience (years): `[1, 2, 3, 6, 9]`.

**Root Mean:** $\bar{y} = (10+20+30+80+90)/5 = 46$

The tree will try **4 candidate thresholds** (midpoints between adjacent experience values):
$t_1 = 1.5,\; t_2 = 2.5,\; t_3 = 4.5,\; t_4 = 7.5$

**Let's evaluate $t_3 = 4.5$ (splits `[10,20,30]` vs `[80,90]`):**

*Left child* `[10, 20, 30]`: mean $= 20$
$$MSE_{left} = \frac{(10-20)^2 + (20-20)^2 + (30-20)^2}{3} = \frac{100+0+100}{3} \approx 66.7$$

*Right child* `[80, 90]`: mean $= 85$
$$MSE_{right} = \frac{(80-85)^2 + (90-85)^2}{2} = \frac{25+25}{2} = 25$$

*Weighted Cost:*
$$Cost(t_3) = \frac{3}{5} \times 66.7 + \frac{2}{5} \times 25 = 40.0 + 10.0 = \mathbf{50.0}$$

**Now let's evaluate $t_2 = 2.5$ (splits `[10,20]` vs `[30,80,90]`) for comparison:**

*Left child* `[10, 20]`: mean $= 15$
$$MSE_{left} = \frac{(10-15)^2 + (20-15)^2}{2} = \frac{25+25}{2} = 25$$

*Right child* `[30, 80, 90]`: mean $= 66.7$
$$MSE_{right} = \frac{(30-66.7)^2 + (80-66.7)^2 + (90-66.7)^2}{3} = \frac{1344.9+176.9+542.9}{3} \approx 688$$

*Weighted Cost:*
$$Cost(t_2) = \frac{2}{5} \times 25 + \frac{3}{5} \times 688 = 10 + 412.8 = \mathbf{422.8}$$

**Conclusion:** $t_3 = 4.5$ wins. The tree confirms that splitting at "Experience > 4.5" is optimal.

**Final prediction rule:** Experience ≤ 4.5 → predict **$20K**. Experience > 4.5 → predict **$85K**.

### **The "Staircase" Shape**
A Regression Tree predicts a constant value in each region — on a plot this is a **step function**. A very deep tree creates hundreds of tiny steps, allowing it to approximate any curve, but also memorizing every data point's noise.

---

## 3. The Curse of Overfitting: Bias-Variance Tradeoff

### **Formal Definitions**

The generalization error of any model can be decomposed into:
$$Total\;Error = Bias^2 + Variance + \text{Irreducible Noise}$$

- **Bias** — How far your model's *average* prediction is from the truth. High bias = the model systematically misses the pattern. (Underfitting)
- **Variance** — How much your model's prediction *changes* if you train it on a different subset of data. High variance = extreme sensitivity to training examples. (Overfitting)
- **Irreducible Noise** — Random noise inherent in the data. No model can remove this.

### **The Dartboard Analogy**
```
High Bias, Low Variance    |  High Bias, High Variance
(Consistently wrong)       |  (Randomly AND consistently wrong)
    . .                    |     .          .
  .     .                  |    .    ⊙      .
  .  ⊙  .     * * *       |     .          .
  .     .    *  *  *       |             *     *
    . .       *  *         |              *  *
              target       |   target (x) predictions (.)

Low Bias, High Variance    |  Low Bias, Low Variance (GOAL)
(Accurate on avg, noisy)   |  (Consistently correct)
    .                      |        * *
  .   .      * * *         |      *  ⊙  *     * * *
   ⊙    .   *  ⊙  *        |        * *      *  ⊙  *
     .      *  *  *        |                 *  *  *
       .                   |
```

### **Where Decision Trees Live**
- **Shallow tree (depth 1-2):** High Bias, Low Variance. Too simple, misses patterns.
- **Deep tree (no limits):** Low Bias, High Variance. Memorizes training noise.

**Key insight for interview:** Decision Trees are fundamentally **High Variance** learners. This is the core motivation for:
- **Random Forest** → Average many independent trees to reduce variance.
- **Gradient Boosting** → Use shallow trees (high bias) and boost them to reduce bias.

---

## 4. Pre-Pruning: Stopping the Tree Early

Pre-pruning means setting constraints *before* building the tree. When a node violates a constraint, it becomes a leaf immediately.

### **A. max_depth**
- **What it controls:** Maximum number of levels from root to any leaf.
- **Mechanic:** When a node reaches `max_depth`, it immediately becomes a leaf — no more splits regardless of impurity.
- **Effect:** Low `max_depth` → High Bias. High `max_depth` → High Variance.
- **Starting point:** `max_depth = 3 to 5` for interpretable trees, `3 to 10` for production.

### **B. min_samples_split**
- **What it controls:** Minimum samples a node must have to be *eligible* to split.
- **Mechanic:** Before attempting any split, check: "Does this node have ≥ `min_samples_split` samples?" If not → leaf.
- **Default:** `min_samples_split=2` (very permissive, often too permissive).

### **C. min_samples_leaf**
- **What it controls:** Minimum samples that *must end up* in each child after a split.
- **Mechanic:** Even if the parent has many samples, a split that sends only 1 sample to the right child is rejected.
- **Example:** `min_samples_leaf=5` → every leaf has ≥ 5 training samples. Predictions never based on a single outlier.

### **D. max_features**
- **What it controls:** How many features are *considered* at each split.
- **Note:** Primarily used inside **Random Forest** to de-correlate trees. For a single tree, reduces performance.

### **Interaction Effect: max_depth + min_samples_leaf**
- `max_depth` controls tree height (global structure).
- `min_samples_leaf` controls leaf size (local granularity).
- Starting recipe: `max_depth=5, min_samples_leaf=10`. Search around these with cross-validation.

---

## 5. Post-Pruning: Cost Complexity Pruning (CCP)

### **The Key Idea**
1. Let the tree grow to **full maximum depth**.
2. Work *backwards*, pruning branches that aren't "worth" their complexity.

### **The Objective Function**
$$Cost_\alpha(T) = R(T) + \alpha \cdot |T|$$

- $R(T)$ — Total training error.
- $|T|$ — Number of leaves (complexity).
- $\alpha$ — The "tax" per leaf. Higher = more aggressive pruning.

### **How to Find the Right Alpha**
1. Grow full tree.
2. Generate pruning path from full tree to a single leaf.
3. Cross-validate to find the $\alpha$ with lowest validation error.

In Scikit-Learn: `ccp_alpha` parameter. Use `cost_complexity_pruning_path(X, y)` to get the full $\alpha$ sequence.

---

## 6. Pros, Cons & Real-World Consequences

### **Pros (With Context)**

**1. Highly Interpretable (White-Box Model)**
- You can export a trained tree as literal business rules: "IF Income > 50k AND Age > 35 → Approve Loan."
- In regulated industries (banking, healthcare), this is often *legally required*.

**2. No Feature Scaling Required**
- Trees use thresholds, not distances. Scale of features is irrelevant.

**3. Non-Parametric — No Distribution Assumptions**
- Makes no assumption about the data's functional form.

### **Cons (With Root Causes)**

**1. High Variance (Instability)**
- Root cause: Greedy algorithm makes irrevocable decisions. One different training sample → entirely different tree.

**2. Axis-Aligned Splits Only**
- Each split compares ONE feature to ONE threshold. Diagonal boundaries require many approximating steps.

**3. Overfitting on Noisy Data**
- Without constraints, a tree will split until every leaf has one sample. Training accuracy = 100%. Test accuracy collapses.

---

## 7. Edge Cases: How to Handle Real-World Data Problems

### **A. Missing Values**
- **Scikit-Learn's CART:** Does NOT handle `NaN`. You must impute first.
- **XGBoost/LightGBM:** Sparsity-aware splitting — learns a default direction (left/right) for missing values at each node.
- **Best practice:** `SimpleImputer(strategy='median')` + add a boolean `feature_was_missing` column.

### **B. Outliers in Features vs. Outliers in Target**
- **Robust to feature outliers:** Splits use ordering (< / >), not absolute value.
- **Sensitive to target outliers (regression):** Leaf prediction = mean. One $10M outlier skews the leaf.
- **Fix:** Use `criterion='absolute_error'` (MAE) in `DecisionTreeRegressor`. Median is more robust than mean.

### **C. Imbalanced Datasets**
1. `class_weight='balanced'` — penalizes minority misclassifications more.
2. SMOTE / undersampling — resample before training.
3. Threshold adjustment — lower decision threshold for minority class.

### **D. High-Cardinality Categorical Features**
1. **Gain Ratio** (C4.5) — penalizes high-fragmentation features.
2. **Target Encoding** — replace category with mean target value.
3. **LightGBM** — native categorical handling without cardinality explosion.

---

## 8. Interpretability: Explaining a Single Tree's Prediction

> **Real-world scenario:** You trained a Decision Tree for fraud detection. A transaction was flagged as fraud. Your manager asks: *"Why did the model flag this specific transaction?"* This section shows you exactly how to answer that question — both technically and in plain business language.

---

### **A. Why Single Trees Are the Gold Standard for Interpretability**
A Decision Tree is the most interpretable supervised ML model that exists. Every prediction follows a **deterministic, traceable path** through the tree — a sequence of if/else conditions, each referencing one feature and one threshold. Unlike:
- **Neural Networks:** Thousands of interacting weights through non-linear activations. No traceable path.
- **Random Forest:** 500 trees vote. No single path explains the consensus.
- **Decision Tree:** ONE path. Every condition that was TRUE along the way IS the complete reason.

---

### **B. Decision Path Tracing — The Mechanism**
For a fraud detection example, suppose your tree looks like:

```
Root: transaction_amount > 5000?
  └── YES → foreign_country = True?
            └── YES → time_of_day < 3am?
                      └── YES → [LEAF: FRAUD] (43/47 samples = 91.5% fraud)
                      └── NO  → [LEAF: LEGIT]
  └── NO  → [LEAF: LEGIT]
```

A transaction arrives: `amount = $8,000`, `foreign = True`, `time = 2:30am`.

**The model's path:**
1. `$8,000 > $5,000` → **TRUE** → go left
2. `foreign = True` → **TRUE** → go left
3. `2:30am < 3am` → **TRUE** → go left → FRAUD

**The explanation (in plain English for a business audience):**
> *"This transaction was flagged as fraud because: (1) the amount [$8,000] exceeded the $5,000 threshold, AND (2) it originated from a foreign country, AND (3) it occurred at 2:30am which is before 3am. Of the 47 training transactions that matched this exact profile, 43 were confirmed fraud (91.5%)."

This is the complete explanation. Three conditions. Fully auditable. Any business stakeholder can understand and challenge it.

---

### **C. Scikit-Learn Tools for Explanation**

#### **1. `export_text()` — The Entire Tree as Business Rules**
```python
from sklearn.tree import export_text
rules = export_text(model, feature_names=['amount', 'foreign', 'time'])
print(rules)
# Output: human-readable if/else rules for ALL paths
# Exportable to PDF for compliance/audit documentation
```

#### **2. `decision_path()` — Trace ONE Sample Programmatically**
```python
node_indicator = model.decision_path(X_sample)   # sparse matrix of visited nodes
leaf_id = model.apply(X_sample)                   # final leaf node ID

# For each visited node, retrieve:
feature = model.tree_.feature[node_id]           # which feature was tested
threshold = model.tree_.threshold[node_id]       # the threshold value
# If feature value <= threshold → went left (TRUE for <= checks)
```
This gives you a **machine-readable audit trail** for each prediction.

#### **3. `plot_tree()` — Visual Explanation**
```python
from sklearn.tree import plot_tree
plot_tree(model, feature_names=feature_names, class_names=['legit', 'fraud'],
          filled=True, impurity=True, proportion=True)
```

**What each field in a plotted node means:**
| Field | Meaning |
| :--- | :--- |
| `amount <= 5000` | Condition tested. Left = TRUE, Right = FALSE. |
| `gini = 0.48` | Impurity of this node. |
| `samples = 120` | Training samples that reached this node. |
| `value = [78, 42]` | Count of [Legit, Fraud] training samples. |
| `class = Fraud` | Majority class (the prediction if this were a leaf). |

---

### **D. Building an Automated Explanation Function**

```python
def explain_prediction(model, X_sample, feature_names):
    """
    Generates a human-readable explanation for ONE sample's prediction.
    """
    node_ids = model.decision_path(X_sample).indices
    explanation = []
    
    for node_id in node_ids[:-1]:  # exclude the leaf itself
        feat_idx = model.tree_.feature[node_id]
        feat_name = feature_names[feat_idx]
        threshold = model.tree_.threshold[node_id]
        value = X_sample[0, feat_idx]
        
        direction = "≤" if value <= threshold else ">"
        explanation.append(
            f"{feat_name} = {value:.2f} {direction} {threshold:.2f}"
        )
    
    prediction = model.predict(X_sample)[0]
    leaf_samples = model.tree_.n_node_samples[node_ids[-1]]
    
    print(f"Prediction: {prediction}")
    print(f"Reason (path taken):")
    for i, condition in enumerate(explanation, 1):
        print(f"  {i}. {condition}")
    print(f"Confidence: Based on {leaf_samples} training samples.")
```

#### **Sample Output for the Fraud Case:**
```
Prediction: fraud
Reason (path taken):
  1. transaction_amount = 8000.00 > 5000.00
  2. is_foreign = 1.00 > 0.50
  3. time_of_day = 2.50 ≤ 3.00
Confidence: Based on 47 training samples.
```

---

### **E. Limitations of Native Tree Interpretability**

1. **Deep trees are unreadable:** A tree with `max_depth=20` technically has a traceable path, but that path may have 20 conditions — too complex for a human to review.
2. **Ensembles have no single path:** A Random Forest with 500 trees has 500 different paths for the same sample. There's no single "reason" — each tree disagrees.

> **What comes next:** For ensembles (Random Forest, XGBoost, LightGBM), we use **SHAP (SHapley Additive exPlanations)** — a game-theory based framework that explains any model's prediction by fairly distributing the contribution of each feature across ALL trees. Covered in Chapter 2.

---

## 9. Interview Deep Dive (10 Questions)

### **Q1: Why doesn't a Decision Tree need feature scaling?**
**A:** Trees split using thresholds — `if feature_X > threshold`. Whether `Income` is in dollars or thousands, the threshold adjusts. No distance ($L_1$, $L_2$) or dot product is ever computed. Compare with KNN (pure distance), SVM (margin in scaled space), Neural Networks (gradient descent scale-sensitive) — all require normalization.

### **Q2: Explain the Bias-Variance Tradeoff using `max_depth` as the dial.**
**A:** `max_depth=1` → High Bias (only one question, ignores most features), Low Variance (nearly identical regardless of training data). `max_depth=None` → Low Bias (zero training error), Extreme High Variance (one different sample reshapes the entire tree). Optimal depth chosen via cross-validation at the test error minimum where $Bias^2 + Variance$ is minimized.

### **Q3: How is MSE used in a Regression Tree and what does a leaf predict?**
**A:** At each node, search for the split minimizing $\frac{N_L}{N} \cdot MSE_L + \frac{N_R}{N} \cdot MSE_R$. The leaf predicts the **mean** of all training samples in that leaf. For a new sample following that path, the constant mean is the only possible output.

### **Q4: What is the difference between pre-pruning and post-pruning?**
**A:** Pre-pruning (e.g., `max_depth=5`) stops early but requires knowing the right depth in advance. Post-pruning (`ccp_alpha`) grows the full tree then removes branches that don't justify their complexity tax $\alpha \cdot |T|$. Post-pruning is more principled but computationally heavier. In practice both are used together.

### **Q5: How would you explain to a non-technical person why a fraud detection tree flagged a transaction?**
**A:** Trace the decision path using `decision_path()`. At each node along the path, record the feature name, threshold, and the actual value. Translate each condition to plain English: "This transaction was flagged because: (1) the amount exceeded $5,000, (2) it was made from a foreign country, and (3) it happened before 3am. Based on historical data, 91.5% of transactions matching this profile were fraud."

### **Q6: What are the Scikit-Learn tools available to explain a Decision Tree's prediction?**
**A:** (1) `export_text()` — prints the full tree as nested if/else rules. (2) `plot_tree()` — visual representation with each node showing condition, impurity, sample count, and class distribution. (3) `decision_path(X)` — returns the sparse path matrix showing which nodes a sample visited. (4) `apply(X)` — returns the leaf node ID for each sample. Together these allow full programmatic explanation of any individual prediction.

### **Q7: When does native tree interpretability break down?**
**A:** Two scenarios: (1) Trees deeper than ~6 levels — the path may have 20+ conditions, too complex for humans to review meaningfully. (2) Ensemble models (Random Forest, XGBoost) — there is no single path. Each of 500 trees takes a different path and votes. No one tree's path is "the" explanation. You need SHAP values for ensemble interpretability.

### **Q8: What happens at the leaf node during classification? During regression?**
**A:** Classification leaf: stores the count of each class (`value = [n_class0, n_class1]`). Prediction = class with highest count. Probability = class count / total samples in leaf. Regression leaf: stores all target values. Prediction = mean of all target values. The leaf is the terminal node — no more questions asked.

### **Q9: How do you handle outliers in target values for a Regression Tree?**
**A:** The default `criterion='squared_error'` (MSE) is sensitive to outliers because squaring amplifies large deviations — one $10M outlier in a $50k-$100k leaf pulls the mean prediction to $1M+. Fix: use `criterion='absolute_error'` (MAE) which uses the **median** of leaf values instead of mean. The median is robust to outliers. However, MAE is slower to compute than MSE.

### **Q10: When should you NOT use a Decision Tree?**
**A:** (1) Linear relationships — approximating a linear boundary requires many splits. Use Linear/Logistic Regression. (2) Small datasets — high variance leads to severe overfitting. Use simpler parametric models. (3) When well-calibrated probabilities are needed — leaf class proportions are often poorly calibrated. Use Platt scaling or isotonic regression. (4) High-dimensional sparse data (NLP) — linear models dominate.

---

## 10. Chapter Summary & Interview Checklist

| # | Question | Minimum Expected Answer |
| :- | :--- | :--- |
| 1 | ID3 vs C4.5 vs CART differences? | Metric, split type, data type support. |
| 2 | Why CART only does binary splits? | Flexibility (reuse feature), computation, continuous feature support. |
| 3 | Regression Tree: how does it split and what does the leaf predict? | Minimize weighted MSE. Leaf predicts mean. |
| 4 | Define Bias and Variance using max_depth as the dial. | Low depth = high bias. High depth = high variance. |
| 5 | What does `min_samples_leaf=10` prevent? | Leaves with tiny sample counts that memorize noise. |
| 6 | What is CCP? What does alpha do? | $Cost = Error + \alpha \cdot |Leaves|$. Alpha = tax per leaf. |
| 7 | Missing values: what does Scikit-Learn do? What does XGBoost do? | CART: must impute. XGBoost: sparsity-aware learned direction. |
| 8 | How do you explain a single tree's fraud prediction to a manager? | Use decision_path() to trace conditions. Translate to plain English. |
| 9 | What Scikit-Learn functions support single-tree interpretability? | export_text(), plot_tree(), decision_path(), apply(). |
| 10 | Why does native interpretability fail for ensembles? | 500 trees, 500 paths. No single explanation. Need SHAP. |

---

## 11. Implementation Playground (Placeholder)

Five separate practice cells — each targeting a specific concept from this chapter.


In [ ]:
# ─── CELL 1: Classification Tree — Iris Dataset ───────────────────────────────
from sklearn.datasets import load_iris
from sklearn.tree import DecisionTreeClassifier, export_text, plot_tree
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report
import matplotlib.pyplot as plt

# TODO: Load Iris, split into train/test (80/20)
pass

# TODO: Train DecisionTreeClassifier with max_depth=3, criterion='gini'
pass

# TODO: Print the tree using export_text() — read the rules and understand each split
pass

# TODO: Visualize using plot_tree() with feature_names and class_names
pass

# TODO: Print classification_report for test set
pass

In [ ]:
# ─── CELL 2: Regression Tree — Salary Prediction ──────────────────────────────
import numpy as np
from sklearn.tree import DecisionTreeRegressor
import matplotlib.pyplot as plt

# Reproduce the worked example from Section 2
experience = np.array([1, 2, 3, 6, 9]).reshape(-1, 1)
salary = np.array([10, 20, 30, 80, 90])

# TODO: Fit a DecisionTreeRegressor with max_depth=1 and criterion='squared_error'
pass

# TODO: Verify the split is at experience~4.5 using model.tree_.threshold
pass

# TODO: Plot prediction curve: x from 0 to 10, predict, overlay as step function
# Compare with actual salary points to see the 'staircase' shape visually
pass

In [ ]:
# ─── CELL 3: Post-Pruning — Finding Optimal ccp_alpha ─────────────────────────
from sklearn.datasets import load_breast_cancer
from sklearn.tree import DecisionTreeClassifier
from sklearn.model_selection import train_test_split
import matplotlib.pyplot as plt

# TODO: Load Breast Cancer dataset, split train/test
pass

# TODO: Get the pruning path using .cost_complexity_pruning_path(X_train, y_train)
pass

# TODO: Train one tree per alpha value; record n_leaves, train_acc, test_acc
pass

# TODO: Plot alpha vs n_leaves (should decrease monotonically)
# TODO: Plot alpha vs train/test accuracy — find test_accuracy peak (sweet spot)
pass

In [ ]:
# ─── CELL 4: Learning Curve — Diagnosing Overfitting vs. Underfitting ─────────
from sklearn.datasets import make_classification
from sklearn.tree import DecisionTreeClassifier
from sklearn.model_selection import learning_curve
import matplotlib.pyplot as plt
import numpy as np

# TODO: Create a synthetic classification dataset with noise
pass

# TODO: Use learning_curve() for deep tree (max_depth=None) vs constrained (max_depth=3)
pass

# TODO: Plot train_score and val_score vs training_size for both models
# Large gap + high train + low test = Overfitting (High Variance)
# Small gap + both low = Underfitting (High Bias)
pass

In [ ]:
# ─── CELL 5: Interpretability — Fraud Detection Decision Path ─────────────────
from sklearn.tree import DecisionTreeClassifier, export_text, plot_tree
from sklearn.datasets import make_classification
from sklearn.model_selection import train_test_split
import numpy as np
import matplotlib.pyplot as plt

feature_names = ['transaction_amount', 'is_foreign', 'time_of_day', 'num_prior_flags']

# TODO: Create a synthetic fraud dataset using make_classification or manual arrays
# with the 4 features above
pass

# TODO: Train a shallow DecisionTreeClassifier (max_depth=4) for interpretability
pass

# TODO: Print the full tree as text rules using export_text()
pass

# TODO: Pick ONE test sample classified as 'fraud'
# Use model.decision_path(X_sample) to get nodes visited
# For each node: print feature name, threshold, sample value, and direction taken
# Produce: "This was classified FRAUD because: 1. amount > X, 2. is_foreign = 1, ..."
pass

# TODO: Implement the explain_prediction() function from Section 8D
# Run it on your fraud sample and compare to the raw decision_path output
pass

# TODO: Visualize with plot_tree(filled=True) and mentally trace your sample's path
pass